# 52 · Human approval for side-effecting tools
Level L5 · Part 6 · professional · ~30 min · builds on 51

## Goal
Put an **approval gate** in Scout's loop: before any `write_file` runs, a human sees the exact call and
approves, edits or rejects it. A rejection comes back to the model as a tool error it can handle, and
every decision is written to an **audit record**. Then watch Gemini handle a rejected and an edited call.

## The idea
Lesson 51 limited *which* tools a task may use. Some calls are allowed but still risky: saving, sending,
paying. For those, put a person in the loop. The loop stops before the call, shows it in full, and waits
for one of three answers: approve, edit the arguments, or reject. Whatever the answer, the model gets a
normal tool result, so the run carries on. And the decision is written down: who, when, what was
proposed, what actually ran. Think of a junior who drafts a letter: the boss signs, corrects or bins
it, and the post room logs every letter that went out.

## Picture
```mermaid
flowchart LR
    M[model asks for a tool] --> Q{read-only?}
    Q -- yes --> T[run tool]
    Q -- no --> H[human sees the full call]
    H -- approve --> T
    H -- edit --> T2[run with new arguments]
    H -- reject --> E[error result: did not run]
    H --> A[(audit record)]
    T & T2 & E --> R[tool result back to the model]
```

## Step by step
### 1. Start the recorder and set up
The recorder first. Without a key, the Gemini calls replay from
`../data/llm_cassettes/52_human-approval.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("52_human-approval")

LLM mode: replay (52_human-approval.yaml)


Imports, the client and the pinned model, as in lesson 51. The audit file is cleared so every run of
this notebook starts fresh.

In [2]:
import hashlib, json, os
from dataclasses import dataclass, asdict
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
AUDIT = Path("../data/audit/52_approvals.jsonl")
AUDIT.parent.mkdir(parents=True, exist_ok=True)
AUDIT.unlink(missing_ok=True)
print("model:", MODEL, "| audit file:", AUDIT)

model: gemini-3.5-flash-lite | audit file: ../data/audit/52_approvals.jsonl


Setup from lesson 26: `ScriptedModel` with its history check.

In [3]:
def check_history(messages: list[dict]) -> None:
    issued = set()
    for m in messages:
        issued |= {c["id"] for c in m.get("tool_calls", [])}
        if m["role"] == "tool" and m["call_id"] not in issued:
            raise ValueError(f"tool result for unknown call id {m['call_id']!r}")

class ScriptedModel:
    def __init__(self, script: list):
        self.script, self.calls, self.next_id = script, 0, 1

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        check_history(messages)
        self.calls += 1
        if self.calls > 50:
            raise RuntimeError("model called 50 times: the loop never stopped")
        reply = self.script[min(self.calls, len(self.script)) - 1]
        reply = reply(messages) if callable(reply) else json.loads(json.dumps(reply))
        for call in reply.get("tool_calls", []):
            call["id"], self.next_id = f"call_{self.next_id}", self.next_id + 1
        return {"role": "assistant", **reply}

Lesson 28's `run_tool`: a crash in a tool becomes an error result.

In [4]:
def run_tool(call: dict, tools: dict) -> dict:
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'.", "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

Lesson 51's loop, unchanged: the tool runner is a parameter, so a gate can be plugged in.

In [5]:
def run_agent(question, model, tools, schemas, max_steps=6, runner=run_tool, log=print) -> dict:
    messages, errors = [{"role": "user", "text": question}], 0
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas)
        messages.append(reply)
        calls = reply.get("tool_calls", [])
        if not calls:
            log(f"step {step}: answer")
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step,
                    "errors": errors, "messages": messages}
        for call in calls:
            result = runner(call, tools)
            errors += result["is_error"]
            log(f"step {step}: {call['name']}({json.dumps(call['args'])[:60]}) -> "
                f"{'ERROR ' if result['is_error'] else ''}{result['content'][:70]}")
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "errors": errors, "messages": messages}

### 2. Scout's notes and three tools
The six notes from lesson 01, `search_notes`, `list_files`, and lesson 29's `write_file`, all as in
lesson 51. `reset` empties Scout's folder and puts back the user's own `mars_minutes.txt`.

In [6]:
NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
SCOUT_DIR = Path("../data/scout_files")
SCOUT_DIR.mkdir(parents=True, exist_ok=True)

def reset() -> None:
    for p in SCOUT_DIR.iterdir():
        p.unlink()
    (SCOUT_DIR / "mars_minutes.txt").write_text("A day on Mars lasts about 1479 minutes.")

The tools and their schemas.

In [7]:
def search_notes(query: str) -> list[dict]:
    """Search the user's notes (case-insensitive). Returns up to 5 matching notes as {id, text}."""
    return [n for n in NOTES if query.lower() in n["text"].lower()][:5]

def list_files() -> list[str]:
    """List the file names in Scout's folder."""
    return sorted(p.name for p in SCOUT_DIR.iterdir())

def write_file(filename: str, text: str, overwrite: bool = False) -> str:
    """Save text to a file in Scout's folder. Refuses an existing file unless overwrite is true."""
    target = (SCOUT_DIR / filename).resolve()
    if target.parent != SCOUT_DIR.resolve():
        raise ValueError(f"'{filename}' is outside Scout's folder. Use a plain file name.")
    if target.exists() and not overwrite:
        raise FileExistsError(f"'{filename}' already exists. Pick a new name or set overwrite to true.")
    target.write_text(text)
    return f"saved {len(text)} characters to {target.name}"

def schema(fn, **params) -> dict:
    required = [p for p in params if p != "overwrite"]
    return {"name": fn.__name__, "description": fn.__doc__, "parameters": {
        "type": "object", "required": required, "properties": {p: {"type": t} for p, t in params.items()}}}

TOOLS = {f.__name__: f for f in (search_notes, list_files, write_file)}
SCHEMAS = [schema(search_notes, query="string"), schema(list_files),
           schema(write_file, filename="string", text="string", overwrite="boolean")]
reset(); print(list(TOOLS), list_files())

['search_notes', 'list_files', 'write_file'] ['mars_minutes.txt']


### 3. The problem: writes happen silently
The user asks for a summary saved to `mars.txt`. The scripted model searches, writes, then answers. Its
last reply reads the `write_file` result, so it says something different when the write failed.

In [8]:
SAVE_Q = "Summarise my notes about Mars and save the summary to mars.txt."

def after_write(messages: list[dict]) -> dict:
    last = messages[-1]
    if last["is_error"]:
        return {"text": f"I did not save the summary. The tool said: {last['content']}"}
    return {"text": f"Done: {json.loads(last['content'])}."}

SAVE_SCRIPT = [
    {"tool_calls": [{"name": "search_notes", "args": {"query": "Mars"}}]},
    {"tool_calls": [{"name": "write_file", "args": {"filename": "mars.txt",
                     "text": "Mars has two moons. A day there lasts about 24 h 39 min."}}]},
    after_write,
]

Run it with lesson 51's plain loop.

In [9]:
reset()
run = run_agent(SAVE_Q, ScriptedModel(SAVE_SCRIPT), TOOLS, SCHEMAS)
print("answer:", run["answer"])
print("files:", list_files())

step 1: search_notes({"query": "Mars"}) -> [{"id": 1, "text": "Mars has two small moons, Phobos and Deimos."}, {"
step 2: write_file({"filename": "mars.txt", "text": "Mars has two moons. A day ) -> "saved 56 characters to mars.txt"
step 3: answer
answer: Done: saved 56 characters to mars.txt.
files: ['mars.txt', 'mars_minutes.txt']


It worked, and nobody was asked. Fine for a summary file; not fine when the call is "overwrite", "email"
or "pay", or when a poisoned note (lesson 51) chose the arguments.

### 4. Which calls need a human?
Write down the **safe** tools, not the risky ones: every tool not on the read-only list needs approval.
A tool added next month is then gated by default. This is **fail closed**.

In [10]:
READ_ONLY = {"search_notes", "list_files"}

def needs_approval(call: dict) -> bool:
    return call["name"] not in READ_ONLY

for name in TOOLS:
    print(f"{name:13} needs approval: {needs_approval({'name': name})}")

search_notes  needs approval: False
list_files    needs approval: False
write_file    needs approval: True


### 5. A decision, and what the human sees
A `Decision` is one of three actions, signed by a reviewer. `describe` turns a call into what the human
reads: every argument **in full** (a reviewer can't approve text they can't see), plus the one fact that
matters most for `write_file`: does it overwrite something? Step 3 left `mars.txt` on disk, so we
`reset()` first; the second call names the user's own `mars_minutes.txt` to show the warning on purpose.

In [11]:
@dataclass
class Decision:
    action: str                 # "approve", "edit" or "reject"
    reviewer: str
    reason: str = ""
    args: dict | None = None    # the new arguments, only for "edit"

def describe(call: dict) -> str:
    lines = [f"Scout wants to run {call['name']}:"]
    lines += [f"  {k}: {v!r}" for k, v in call["args"].items()]
    if call["name"] == "write_file":
        exists = (SCOUT_DIR / call["args"].get("filename", "?")).is_file()
        lines.append("  -> OVERWRITES an existing file" if exists else "  -> creates a new file")
    return "\n".join(lines)

reset()
mars_call = SAVE_SCRIPT[1]["tool_calls"][0]
print(describe(mars_call))
print(describe({**mars_call, "args": {**mars_call["args"], "filename": "mars_minutes.txt"}}))

Scout wants to run write_file:
  filename: 'mars.txt'
  text: 'Mars has two moons. A day there lasts about 24 h 39 min.'
  -> creates a new file
Scout wants to run write_file:
  filename: 'mars_minutes.txt'
  text: 'Mars has two moons. A day there lasts about 24 h 39 min.'
  -> OVERWRITES an existing file


### 6. The audit record
One JSON line per decision, appended, never edited. It holds what the model **proposed**, what actually
**ran**, who decided, why, and when. `args_sha256` is a fingerprint of the arguments that ran, so you can
later show that the file written matches what was approved. The clock is fixed so outputs repeat.

In [12]:
def fingerprint(args: dict) -> str:
    return hashlib.sha256(json.dumps(args, sort_keys=True).encode()).hexdigest()[:12]

def fixed_clock(start=0):
    tick = iter(range(start, 10_000))
    return lambda: f"2026-10-09T10:00:{next(tick):02d}Z"

def write_audit(record: dict, path: Path = AUDIT) -> None:
    with path.open("a") as f:
        f.write(json.dumps(record) + "\n")

### 7. The approval gate
The gate is a tool runner, so it plugs into the loop like lesson 51's `Guard`. Read-only calls pass
straight through. For the rest it asks the `approver`, writes the audit record **before** running
anything, then acts. Rejected and unknown answers never run the tool: they become an error result
(lesson 28) that tells the model not to retry. An edit is reported to the model, so it doesn't claim
the file it proposed was saved.

In [13]:
class ApprovalGate:
    def __init__(self, approver, run_id: str, runner=run_tool, clock=None):
        self.approver, self.run_id, self.runner = approver, run_id, runner
        self.clock = clock or fixed_clock()

    def __call__(self, call: dict, tools: dict) -> dict:
        if not needs_approval(call):
            return self.runner(call, tools)
        d = self.approver(call)
        ran = d.action in ("approve", "edit")
        final = (d.args if d.action == "edit" else call["args"]) if ran else None
        write_audit({"ts": self.clock(), "run_id": self.run_id, "call_id": call["id"], "tool": call["name"],
                     "proposed": call["args"], "decision": d.action, "final_args": final,
                     "args_sha256": fingerprint(final) if ran else None,
                     "reviewer": d.reviewer, "reason": d.reason})
        if not ran:
            return {"content": f"rejected by the user ({d.reason or 'no reason given'}). The call did not "
                               "run. Do not retry it; tell the user.", "is_error": True}
        result = self.runner({**call, "args": final}, tools)
        if d.action == "edit" and not result["is_error"]:
            result["content"] = json.dumps({"edited_by_user": final, "output": json.loads(result["content"])})
        return result

### 8. Approve
A scripted approver stands in for the human: it hands out prepared decisions in order and prints what
the human would have seen.

In [14]:
def scripted_approver(*decisions: Decision):
    queue = list(decisions)
    def approve(call: dict) -> Decision:
        print(describe(call)); d = queue.pop(0); print(f"  [{d.reviewer}: {d.action}]")
        return d
    return approve

reset()
gate = ApprovalGate(scripted_approver(Decision("approve", "ana")), run_id="run-approve")
run = run_agent(SAVE_Q, ScriptedModel(SAVE_SCRIPT), TOOLS, SCHEMAS, runner=gate)
print("answer:", run["answer"], "| files:", list_files())

step 1: search_notes({"query": "Mars"}) -> [{"id": 1, "text": "Mars has two small moons, Phobos and Deimos."}, {"
Scout wants to run write_file:
  filename: 'mars.txt'
  text: 'Mars has two moons. A day there lasts about 24 h 39 min.'
  -> creates a new file
  [ana: approve]
step 2: write_file({"filename": "mars.txt", "text": "Mars has two moons. A day ) -> "saved 56 characters to mars.txt"
step 3: answer
answer: Done: saved 56 characters to mars.txt. | files: ['mars.txt', 'mars_minutes.txt']


Same result as step 3, but a person saw the exact text first and the audit file has a line for it.

### 9. Edit
Ana prefers a different file name. The edited call runs, the tool's own checks still apply (an edit can
be wrong too), and the model's result says what changed.

In [15]:
reset()
edited = {"filename": "mars_summary.txt", "text": "Mars has two moons. A day there lasts about 24 h 39 min."}
gate = ApprovalGate(scripted_approver(Decision("edit", "ana", "keep summaries in *_summary.txt", edited)),
                    run_id="run-edit", clock=fixed_clock(10))
run = run_agent(SAVE_Q, ScriptedModel(SAVE_SCRIPT), TOOLS, SCHEMAS, runner=gate, log=lambda *a: None)
print("answer:", run["answer"])
print("files:", list_files())

Scout wants to run write_file:
  filename: 'mars.txt'
  text: 'Mars has two moons. A day there lasts about 24 h 39 min.'
  -> creates a new file
  [ana: edit]
answer: Done: {'edited_by_user': {'filename': 'mars_summary.txt', 'text': 'Mars has two moons. A day there lasts about 24 h 39 min.'}, 'output': 'saved 56 characters to mars_summary.txt'}.
files: ['mars_minutes.txt', 'mars_summary.txt']


### 10. Reject
Now Ana rejects the call. Nothing is written, the loop does not crash, and the model's next turn reads
the error and tells the user. That is the whole point of turning a "no" into a tool error. (No `reset()`
here: step 9's file stays on disk for the audit check in step 12.)

In [16]:
gate = ApprovalGate(scripted_approver(Decision("reject", "ana", "do not save anything today")),
                    run_id="run-reject", clock=fixed_clock(20))
run = run_agent(SAVE_Q, ScriptedModel(SAVE_SCRIPT), TOOLS, SCHEMAS, runner=gate)
print("answer:", run["answer"])
print("mars.txt written:", (SCOUT_DIR / "mars.txt").exists(), "| errors in run:", run["errors"])

step 1: search_notes({"query": "Mars"}) -> [{"id": 1, "text": "Mars has two small moons, Phobos and Deimos."}, {"
Scout wants to run write_file:
  filename: 'mars.txt'
  text: 'Mars has two moons. A day there lasts about 24 h 39 min.'
  -> creates a new file
  [ana: reject]
step 2: write_file({"filename": "mars.txt", "text": "Mars has two moons. A day ) -> ERROR rejected by the user (do not save anything today). The call did not ru
step 3: answer
answer: I did not save the summary. The tool said: rejected by the user (do not save anything today). The call did not run. Do not retry it; tell the user.
mars.txt written: False | errors in run: 1


### 11. A real prompt for a person
In a terminal app the approver reads the keyboard. `console_approver` takes `ask` (normally `input`)
so we can feed it keystrokes here. Anything other than `a` or `e` counts as a reject: fail closed again.

In [17]:
def console_approver(ask=input, reviewer="you"):
    def approve(call: dict) -> Decision:
        print(describe(call))
        choice = ask("[a]pprove, [e]dit file name, [r]eject? ").strip().lower()[:1]
        if choice == "a":
            return Decision("approve", reviewer)
        if choice == "e":
            new_name = ask("new file name: ").strip()
            return Decision("edit", reviewer, "renamed", {**call["args"], "filename": new_name})
        return Decision("reject", reviewer, ask("reason: ").strip())
    return approve

A fake keyboard that types `x` (a typo) and a reason. The typo is treated as a reject.

In [18]:
keys = iter(["x", "typo, but safe"])
def keyboard(prompt: str) -> str:
    key = next(keys); print(prompt + key); return key

gate = ApprovalGate(console_approver(ask=keyboard, reviewer="ben"), run_id="run-console", clock=fixed_clock(30))
run = run_agent(SAVE_Q, ScriptedModel(SAVE_SCRIPT), TOOLS, SCHEMAS, runner=gate, log=lambda *a: None)
print("answer:", run["answer"])

Scout wants to run write_file:
  filename: 'mars.txt'
  text: 'Mars has two moons. A day there lasts about 24 h 39 min.'
  -> creates a new file
[a]pprove, [e]dit file name, [r]eject? x
reason: typo, but safe
answer: I did not save the summary. The tool said: rejected by the user (typo, but safe). The call did not run. Do not retry it; tell the user.


### 12. Read the audit trail
Every decision from steps 8 to 11, one line each. This is what you hand to whoever asks "who let Scout
write that file?"

In [19]:
records = [json.loads(line) for line in AUDIT.read_text().splitlines()]
for r in records:
    print(f"{r['ts']}  {r['run_id']:12} {r['tool']:10} {r['decision']:7} by {r['reviewer']:4} "
          f"ran={r['final_args'] and r['final_args']['filename']}  sha={r['args_sha256']}  {r['reason']}")

2026-10-09T10:00:00Z  run-approve  write_file approve by ana  ran=mars.txt  sha=c2a73fba5bf4  
2026-10-09T10:00:10Z  run-edit     write_file edit    by ana  ran=mars_summary.txt  sha=e858e79b04e7  keep summaries in *_summary.txt
2026-10-09T10:00:20Z  run-reject   write_file reject  by ana  ran=None  sha=None  do not save anything today
2026-10-09T10:00:30Z  run-console  write_file reject  by ben  ran=None  sha=None  typo, but safe


The fingerprint lets you check a record against what happened. For the last call that ran (step 9's
edit), recompute it from the arguments that actually ran.

In [20]:
last = [r for r in records if r["final_args"]][-1]
print("matches the audit record:", fingerprint(last["final_args"]) == last["args_sha256"])
print("file on disk matches the approved text:",
      (SCOUT_DIR / last["final_args"]["filename"]).read_text() == last["final_args"]["text"])

matches the audit record:

 True
file on disk matches the approved text: True


### 13. Gemini meets a rejection
Lesson 51's Gemini wrapper and `to_contents`, unchanged. Error results go back as `{"error": ...}` inside
a `function_response`.

In [21]:
def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])                          # keeps the thought signature
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": json.loads(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(
                id=m["call_id"], name=m["name"], response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

The model wrapper.

In [22]:
SYSTEM = "You are Scout. Answer questions about the user's notes. Search the notes first. Answer in two sentences."

class GeminiModel:
    def __init__(self, system: str = SYSTEM):
        self.system = system

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        decls = [types.Tool(function_declarations=[types.FunctionDeclaration(
            name=s["name"], description=s["description"], parameters_json_schema=s["parameters"]) for s in tools])]
        cfg = types.GenerateContentConfig(system_instruction=self.system, temperature=0, max_output_tokens=300,
            tools=decls, automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
        resp = client.models.generate_content(model=MODEL, contents=to_contents(messages), config=cfg)
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        reply = {"tool_calls": calls} if calls else {"text": resp.text}
        return {"role": "assistant", **reply, "raw": resp.candidates[0].content}

Gemini gets the same request, and Ana rejects whatever it tries to write (the approver hands out the
same decision every time). If this cell is skipped (no key, no recording), step 10 already showed the
mechanics with the fake.

In [23]:
reset()
always_reject = lambda call: (print(describe(call)), Decision("reject", "ana", "do not save anything today"))[1]
gate = ApprovalGate(always_reject, run_id="run-gemini-reject", clock=fixed_clock(50))
run = run_agent(SAVE_Q, GeminiModel(), TOOLS, SCHEMAS, runner=gate)
print("answer:", run["answer"])
print("files:", list_files())

step 1: search_notes({"query": "Mars"}) -> [{"id": 1, "text": "Mars has two small moons, Phobos and Deimos."}, {"
Scout wants to run write_file:
  filename: 'mars.txt'
  overwrite: True
  text: 'Mars features two small moons, Phobos and Deimos, and a day that lasts approximately 24 hours and 39 minutes. Additionally, the rover Perseverance successfully landed on the planet in February 2021.'
  -> creates a new file
step 2: write_file({"filename": "mars.txt", "overwrite": true, "text": "Mars fe) -> ERROR rejected by the user (do not save anything today). The call did not ru
step 3: answer
answer: Mars has two small moons, Phobos and Deimos, and a day that lasts approximately 24 hours and 39 minutes, alongside the landing of the Perseverance rover in February 2021. However, I was unable to save the summary to mars.txt because saving files was rejected by the user.
files: ['mars_minutes.txt']


In this recording Gemini made one `write_file` call, read the rejection and did not retry: its answer
gives the summary in the chat and says the file was not saved. Look at what Ana was shown, too: Gemini
set `overwrite: True` though nobody asked for it. The file didn't exist, so it was harmless here, but
this is exactly what a reviewer must be able to see. This is one recorded run; another model might
retry, and then the gate asks again each time until the step limit (lesson 30).

Now an edit: Ana renames the file. Does Gemini report the name that was actually used?

In [24]:
reset()
def rename(call):
    print(describe(call))
    return Decision("edit", "ana", "keep summaries in *_summary.txt", {**call["args"], "filename": "mars_summary.txt"})
gate = ApprovalGate(rename, run_id="run-gemini-edit", clock=fixed_clock(60))
run = run_agent(SAVE_Q, GeminiModel(), TOOLS, SCHEMAS, runner=gate)
print("answer:", run["answer"])
print("files:", list_files())

step 1: search_notes({"query": "Mars"}) -> [{"id": 1, "text": "Mars has two small moons, Phobos and Deimos."}, {"
Scout wants to run write_file:
  text: 'Mars has two small moons named Phobos and Deimos, and a day lasts about 24 hours and 39 minutes. Additionally, the rover Perseverance successfully landed on the planet in February 2021.'
  filename: 'mars.txt'
  overwrite: True
  -> creates a new file
step 2: write_file({"text": "Mars has two small moons named Phobos and Deimos, ) -> {"edited_by_user": {"text": "Mars has two small moons named Phobos and
step 3: answer
answer: Your notes about Mars indicate that it has two small moons, Phobos and Deimos, and a day that lasts about 24 hours and 39 minutes. Additionally, the Perseverance rover landed on the planet in February 2021.
files: ['mars_minutes.txt', 'mars_summary.txt']


The edit worked: `mars_summary.txt` exists and `mars.txt` does not. But Gemini's answer gives the summary
and never says where it was saved, even though the result reported the edit. So don't rely on the model
to tell the user what a human changed: show the audit record in your UI instead. Note too that the
rename kept Gemini's `overwrite: True`; an edit carries every argument you don't change.

### 14. Clean up
Put Scout's folder back to just the user's file.

In [25]:
reset()
print(list_files(), "| audit lines:", len(AUDIT.read_text().splitlines()))

['mars_minutes.txt'] | audit lines: 6


## At work
- **Protocols ask for it.** The MCP tools spec: there "SHOULD always be a human in the loop with the
  ability to deny tool invocations"; clients show tool inputs first and "log tool usage for audit
  purposes". OWASP LLM01 lists approval for high-risk actions. → track agentic-ai-4-protocols
- **Real apps pause, not block.** The OpenAI Agents SDK has `needs_approval` on tools, stops with
  `interruptions`, saves a `RunState` and resumes after `approve` / `reject`, so a reviewer can answer
  an hour later. Saved state must be kept server-side and checked. → track agentic-ai-6-production
- **Tool hints are not policy.** MCP says annotations such as "read-only" are untrusted unless the
  server is; decide what needs approval in your own code. → track agentic-ai-4-protocols
- **Shipping it.** Scout's production checkpoint keeps this gate on every write. → lesson 55

## What just happened
- Without a gate, Scout wrote `mars.txt` and nobody was asked.
- `needs_approval` gates every tool not on the read-only list, so new tools are gated by default.
- The gate showed the full call, then approved, edited (`mars_summary.txt`) or rejected it; a rejection
  became an error result and the scripted model told the user nothing was saved.
- A typo at the console prompt counted as a reject.
- The audit file holds one line per decision, with a fingerprint that matched the file on disk.
- Gemini, in this recording, accepted the rejection without retrying; after the edit it saved to the new
  name but did not mention it, and both times it asked for `overwrite: True` unprompted.

## Common mistakes
**1. Raising on a rejection.** It feels natural to `raise PermissionError`. But the loop does not catch
runner errors, so one "no" kills the whole run and the model never gets to answer.

In [26]:
def strict_gate(call, tools):
    if needs_approval(call):
        raise PermissionError(f"user rejected {call['name']}")
    return run_tool(call, tools)

reset()
run_agent(SAVE_Q, ScriptedModel(SAVE_SCRIPT), TOOLS, SCHEMAS, runner=strict_gate, log=lambda *a: None)

PermissionError: user rejected write_file

Fix: return an error result, as `ApprovalGate` does. The run finishes and the user gets an answer.

In [27]:
reset()
gate = ApprovalGate(lambda call: Decision("reject", "ana", "not now"), run_id="run-fix-1", clock=fixed_clock(70))
print(run_agent(SAVE_Q, ScriptedModel(SAVE_SCRIPT), TOOLS, SCHEMAS, runner=gate, log=lambda *a: None)["answer"])

I did not save the summary. The tool said: rejected by the user (not now). The call did not run. Do not retry it; tell the user.


**2. Listing the risky tools instead of the safe ones.** Someone adds `delete_file` next month and
forgets to add it to `RISKY`. It now runs without asking.

In [28]:
RISKY = {"write_file"}
def needs_approval_open(call): return call["name"] in RISKY

new_call = {"id": "call_9", "name": "delete_file", "args": {"filename": "mars_minutes.txt"}}
assert needs_approval_open(new_call), "delete_file runs without asking anyone"

AssertionError: delete_file runs without asking anyone

Fix: list what is safe, gate everything else (step 4). The new tool is gated before anyone remembers it.

In [29]:
print("delete_file needs approval:", needs_approval(new_call))

delete_file needs approval: True


## Production checklist
- [ ] Every tool not explicitly read-only needs approval (fail closed); the list lives in code, not in tool hints.
- [ ] The reviewer sees every argument in full, plus the consequence (overwrite, recipient, amount).
- [ ] Approve, edit and reject all return a tool result; rejections say "did not run, do not retry".
- [ ] Unknown answers and missing decisions count as reject.
- [ ] The audit record is written before the tool runs: run id, call id, proposed and final arguments, reviewer, reason, time.
- [ ] The step limit still applies, so a model that keeps retrying a rejected call stops.

## Try it
Reviewers get tired of approving the same tool ten times in one run. Add an `approve_all` choice: when
the approver returns `Decision("approve_all", ...)`, later calls to the **same tool in the same run** are
approved without asking, but still audited, with reviewer `"auto (approve_all by ana)"`. Test it with a
scripted model that writes three new files in one run: the approver should be asked once, and the audit
file should get three lines.

In [30]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [31]:
llm_replay.stop()

## Key terms
- **human in the loop** — a person must act at a set point in an automated run, here before a risky tool call runs.
- **approval gate** — a tool runner that holds a side-effecting call until a human approves, edits or rejects it.
- **audit record** — an append-only entry saying who decided what about which call, when, and what actually ran.
- **fail closed** — when unsure (a new tool, a typo, a missing decision), deny rather than allow.